In [20]:
import gymnasium as gym
import numpy as np
import pandas as pd
import math

In [21]:
class TrackEnv(gym.Env):
  def __init__(self):


    self.matrix = buildTrack()

    #self.width = width
    #self.height = height

    self.view_size = 7
    self.road_width = 5



    coordinates = np.argwhere(self.matrix == 2)
    print(self.matrix)
    coordinates = coordinates[:, [1, 0]]
    print(f"coordinate{coordinates}")



    self._target_location = np.array(coordinates, dtype = np.int32)


    #da cambiare
    self._agent_location = np.array(coordinates[self.road_width // 2], dtype = np.int32)
    print(f"coordinate pilota{self._agent_location}")
    #capire come mettere direzione dinamica in base alla pista
    #self._agent_direction = math.pi
    self._agent_velocity = 1



    #COME VOGLIAMO MODELLARE IL NOSTRO AGENTE? VOGLIAMO CHE LE SUE AZIONE SIANO
    # SU, GIU, DESTRA E SINISTRA OPPURE DEL TIPO ACELLERA, FRENA, GIRA A DESTRA
    #E GIRA A SINISTRA

    #azione 0 è frena
    #azione 1 è accellera
    #azione 2 è gira a destra
    #azione 3 è gira a sinistra
    self.action_space = gym.spaces.Discrete(4)

    self._action_to_direction = {
            0: np.array([0, 1]),   # Move right (column + 1)
            1: np.array([-1, 0]),  # Move up (row - 1)
            2: np.array([0, -1]),  # Move left (column - 1)
            3: np.array([1, 0]),   # Move down (row + 1)
        }

    # SE VOLESSIMO USARE IL SECONDO TIPO DI AZIONI DOBBIAMO ANCHE DIRE ALL'AGENTE
    # IN CHE DIREZIONE STA ANDANDO, E QUINDI AGGIUNGERE UN VALORE ALLO STATO
    # POTREMMO FARE CHE LA DIREZIONE è SEMPLICEMENTE UN NUMERO FLOAT CHE INDICA
    # L'ANGOLO TRA UN IMMAGINARIA LINEA VERTICALE CHE PASSA PER IL PILOTA E LA
    # DIREZIONE EFFETTIVA DEL PILOTA
    # ESEMPIO SPAZIO OSSERVAZIONI FATTO DA CHAT

    self.observation_space = gym.spaces.Dict({  #i dict servono per spazi eterogenei
            # 1. La sottomatrice locale (come prima)
            "agent_view": gym.spaces.Box( #i box servono per spazi omogenei
                low=-2,
                high=2,
                shape=(self.view_size, self.view_size),
                dtype=np.int8
            ),

            # 2. La direzione (angolo in radianti o gradi)
            # Definiamo un array di shape=(1,) contenente un float
            # Esempio: da 0 a 2*PI (circa 6.28)
            #"direction": gym.spaces.Box(
            #    low=0,
            #    high=2 * math.pi,
            #    shape=(1,),
            #    dtype=np.float32
            #),

            #aggiungere velocità
            "velocity": gym.spaces.Box(
                low=0,
                high=self.matrix.shape[0],
                shape=(1,),
                dtype=np.float32
            )
        })










  def _get_obs(self):
    view_padding = self.view_size // 2
    max_x,max_y = self.matrix.shape

    # HO CAMBIAMO IL LOW DELLO SPAZIO DELLE OSSERVAZIONI DA -1 A -2 PERCHè SERVE
    # UN MODO PER INDICARE UNA CELLA CHE NON ESISTE

    # creo la matrice che vede il pilota
    view_matrix = np.matrix(np.ones(shape=(self.view_size, self.view_size)))
    # inizializzo tutte le celle viste dal pilota come non esistenti
    for i in range(self.view_size):
      view_matrix[i]=view_matrix[i]*-2
    #indice della prima cella (top-left) della matrice della vista pilota nella matrice grande
    tl_x = self._agent_location[0] - view_padding
    tl_y = self._agent_location[1] - view_padding

    view_matrix_x = range(tl_x, tl_x+self.view_size,1)
    view_matrix_y = range(tl_y, tl_y+self.view_size,1)

    #tengo soltanto gli indici interni alla matrice grande
    clipped_view_matrix_x = np.clip(view_matrix_x,0, max_x - 1)
    clipped_view_matrix_y = np.clip(view_matrix_y,0, max_y - 1)
    #rimuovo i duplicati creati da np.clip
    clipped_view_matrix_x = list(set(clipped_view_matrix_x))
    clipped_view_matrix_y = list(set(clipped_view_matrix_y))

    #copio i valori dalla matrice grande in quella della visione del pilota
    for x in clipped_view_matrix_x:
      for y in  clipped_view_matrix_y:
        view_matrix[x-tl_x , y-tl_y]=self.matrix[y,x]

    view_matrix = view_matrix.T


    return { "agent_view": view_matrix ,
            #"direction": self._agent_direction,
             "velocity":self._agent_velocity}











  #come info restituiamo la posizione e direzione del pilota
  def _get_info(self):
    return { "agent_location":self._agent_location,
            #"agent_direction":self._agent_direction,
             "agent_velocity":self._agent_velocity}









  #con reset, rimettiamo il pilora nella posizione iniziale dal traguard
  # e ristabiliamo la sua direzione
  def reset(self, seed = None, options = None):

    super().reset(seed=seed)

   # self._agent_direction = math.pi

    self._agent_velocity = 1

    coordinates = np.argwhere(self.matrix == 2)

    coordinates = coordinates[:, [1, 0]]
    #da cambiare
    self._agent_location = np.array(coordinates[self.road_width // 2], dtype = np.int32)


    observation = self._get_obs()
    info = self._get_info()
    return observation, info







  def step(self, action):

    size = self.matrix.shape[0]
    direction = self._action_to_direction[action]

        # Update agent position, ensuring it stays within grid bounds
        # np.clip prevents the agent from walking off the edge
    self._agent_location = np.clip(
            self._agent_location + direction, 0, size - 1
        )

        # Check if agent reached the target
    if self._target_location.count(self._agent_location, self._target_location) == 1:
       terminated = True

        # We don't use truncation in this simple environment
        # (could add a step limit here if desired)
    truncated = False
    # traguardo si attiva quando tutti i checkpoint sono stati traversati
    # se il tragurdo è traversato prima gli diamo un reward negativo assurdo
    # da aggingere logica reward negativo fuori strada
    #reward = distanza
    #reward = -1 costante + bonus checkpoint
    #reward = -(distanza_massima - distanza) costante checkpoint 2 volte -> reward negativo
    # reward = -( numero_reward_traguardo - distnza) costante + logica checkpoint
    # reward =

    reward = 1 if terminated else 0

    observation = self._get_obs()
    info = self._get_info()

    return observation, reward, terminated, truncated, info





    return 0









def buildTrack(fileName = "/imola_track.csv"):
  df = pd.read_csv(fileName, header = None
                   #,sep=';'
                   )
  matrice_circuito = df.to_numpy()
  print(f"Dimensioni matrice:{matrice_circuito.shape}")
  #print(matrice_circuito)
  return matrice_circuito







In [22]:
env = TrackEnv()
env.reset()

Dimensioni matrice:(60, 60)
[[-1 -1 -1 ... -1 -1 -1]
 [-1 -1 -1 ... -1 -1 -1]
 [-1 -1 -1 ... -1 -1 -1]
 ...
 [-1 -1 -1 ... -1 -1 -1]
 [-1 -1 -1 ... -1 -1 -1]
 [-1 -1 -1 ... -1 -1 -1]]
coordinate[[35 46]
 [35 47]
 [35 48]
 [35 49]
 [35 50]
 [35 51]
 [35 52]
 [35 53]
 [35 54]]
coordinate pilota[35 48]


({'agent_view': matrix([[-1., -1., -1., -1., -1., -1., -1.],
          [ 0.,  0.,  0.,  2., -1., -1., -1.],
          [ 1.,  1.,  1.,  2.,  0.,  0.,  0.],
          [ 1.,  1.,  1.,  2.,  1.,  1.,  1.],
          [ 1.,  1.,  1.,  2.,  1.,  1.,  1.],
          [ 1.,  1.,  1.,  2.,  1.,  1.,  1.],
          [ 1.,  1.,  1.,  2.,  1.,  1.,  1.]]),
  'velocity': 1},
 {'agent_location': array([35, 48], dtype=int32), 'agent_velocity': 1})

Prototipo funzione che calcola la distanza, lo so che è orribile ma era soltanto per capire se il metodo della coda funzionava.
NOTA: è POSSIBILE CHE COORDINATE NON CONTENGA LE EFFETTIVE COORDINATE DEL TRAGUARDO

In [23]:

def creaMatriceDistanze( nomeFile, direzione):
  df = pd.read_csv(nomeFile, header = None)
  matrice_distanze = df.to_numpy()

  larghezza, altezza=matrice_distanze.shape

  for x in range(larghezza):
    for y in range(altezza):
      if matrice_distanze[x,y] != -1:
        matrice_distanze[x,y] = -2
  print(altezza)
  print(larghezza)

  a = matrice_distanze

  traguardo = list([[37, 51], [37, 52],[37 ,53],[37 ,54],[37, 55], [37, 56], [37, 57], [37, 58]]) # centrato il traguardo? inclusi i cordoli
  index = traguardo.copy() #secondo me ha più senso calcolare la distanza dalla prima cella del traguardo disponibile rispetto ad una (list([[35,50]]))
  a[37, 51]=0
  a[37, 52]=0
  a[37 ,53]=0
  a[37 ,54]=0
  a[37, 55]=0
  a[37, 56]=0
  a[37, 57]=0
  a[37, 58]=0


  match direzione:
    case "destra":
      slider=[0,1]
    case "sinistra":
      slider=[0,-1]
    case "basso":
      slider=[1,0]
    case "alto":
      slider=[-1,0]
    case _:
      slider=[0,0]


  for i in index:
    i[0]=i[0]+slider[0]
    i[1]=i[1]+slider[1]
    a[i[0],i[1]]=1
  print(index)

  while len(index) != 0:
      i = index.pop(0)
      #bisogna capire come non fare andare all'indietro del traguardo
      if (i[1]-1) >= 0 and a[i[0],i[1]-1] == -2 and traguardo.count([i[0],i[1]-1]) == 0:
          a[i[0],i[1]-1] = a[i[0],i[1]]+1
          index.append([i[0],i[1]-1])

      if (i[1]+1) < altezza and a[i[0],i[1]+1] == -2  and traguardo.count([i[0],i[1]+1]) == 0:
          a[i[0],i[1]+1] = a[i[0],i[1]]+1
          index.append([i[0],i[1]+1])

      if (i[0]-1) >= 0 and a[i[0]-1,i[1]] == -2  and traguardo.count([i[0]-1,i[1]]) == 0:
          a[i[0]-1,i[1]] = a[i[0],i[1]]+1
          index.append([i[0]-1,i[1]])

      if (i[0]+1) < larghezza and a[i[0]+1,i[1]] == -2  and traguardo.count([i[0]+1,i[1]]) == 0:
          a[i[0]+1,i[1]] = a[i[0],i[1]]+1
          index.append([i[0]+1,i[1]])

  # 2. Convertiamo in DataFrame
  df = pd.DataFrame(a)

  # 3. Salviamo in CSV
  df.to_csv('mio_file.csv', index=False, header=False)

  return True

print(creaMatriceDistanze("/imola_track.csv","alto"))

60
60
[[36, 51], [36, 52], [36, 53], [36, 54], [36, 55], [36, 56], [36, 57], [36, 58]]
True
